# Homework

Download the model files

```bash
PREFIX="https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
DATA_URL="${PREFIX}/hair_classifier_v1.onnx.data"
MODEL_URL="${PREFIX}/hair_classifier_v1.onnx"
wget ${DATA_URL}
wget ${MODEL_URL}
```

In [4]:
import onnxruntime as ort

onnx_model_path = "models/hair_classifier_v1.onnx"
session = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])

## Question 1

What's the name of the output:

- output
- sigmoid
- softmax
- prediction

In [5]:
inputs = session.get_inputs()
outputs = session.get_outputs()

input_name = inputs[0].name
output_name = outputs[0].name

print("Input name:", input_name)
print("Output name:", output_name)

Input name: input
Output name: output


A/ `output`

## Preparing the image

In [6]:
from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

## Question 2: Target size

Let's download and resize this image:

https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

Based on the previous homework, what should be the target size for the image?

- 64x64
- 128x128
- 200x200
- 256x256

In [10]:
img = download_image('https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg')
img = prepare_image(img, (200, 200))

A/ `200x200`

## Question 3

After the pre-processing, what's the value in the first pixel, the R channel?

> Tip: Check the previous homework. What was the pre-processing we did there?

- -10.73
- -1.073
- 1.073
- 10.73


In [26]:
import numpy as np
from keras_image_helper import create_preprocessor

In [ ]:
def preprocess_pytorch(X):
    # X: shape (1, 299, 299, 3), dtype=float32, values in [0, 255]
    X = X / 255.0

    mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

    # Convert NHWC → NCHW
    # from (batch, height, width, channels) → (batch, channels, height, width)
    X = X.transpose(0, 3, 1, 2)

    # Normalize
    X = (X - mean) / std

    return X.astype(np.float32)


preprocessor = create_preprocessor(preprocess_pytorch, target_size=(200, 200))


In [29]:
X = preprocessor.from_url("https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg")

In [30]:
X

array([[[[-1.0732939 , -1.0047948 , -1.0390444 , ..., -1.0732939 ,
          -1.0732939 , -1.210292  ],
         [-1.0561692 , -1.0219196 , -1.0047948 , ..., -1.0219196 ,
          -1.0561692 , -1.1760424 ],
         [-0.9534206 , -0.97054535, -0.9191711 , ..., -1.0219196 ,
          -1.1075435 , -1.2274168 ],
         ...,
         [-1.6726604 , -1.6726604 , -1.5185375 , ...,  1.7351657 ,
           1.649542  ,  1.7865399 ],
         [-1.6726604 , -1.6555356 , -1.6384108 , ...,  1.6837914 ,
           1.5981677 ,  1.7009162 ],
         [-1.6384108 , -1.7240345 , -1.6726604 , ...,  1.718041  ,
           1.7351657 ,  1.6837914 ]],

        [[-0.21498597, -0.10994396, -0.10994396, ..., -0.512605  ,
          -0.477591  , -0.635154  ],
         [-0.19747896, -0.16246496, -0.16246496, ..., -0.512605  ,
          -0.547619  , -0.60014004],
         [-0.07492995, -0.12745096, -0.14495796, ..., -0.477591  ,
          -0.60014004, -0.705182  ],
         ...,
         [-1.1428571 , -1.1428571 

A/ `-1.073`

## Question 4

Now let's apply this model to this image. What's the output of the model?

- 0.09
- 0.49
- 0.69
- 0.89


In [32]:
result = session.run([output_name], { input_name: X})

In [35]:
predictions = result[0][0].tolist()

In [36]:
print(predictions)

[0.0915662869811058]


A/ `0.09`

## Prepare the lambda code

Now you need to copy all the code into a separate python file. You will need to use this file for the next two questions.

## Docker

For the next two questions, we'll use a Docker image that we already prepared. This is the Dockerfile that we used for creating the image:

```dockerfile
FROM public.ecr.aws/lambda/python:3.13

COPY hair_classifier_empty.onnx.data .
COPY hair_classifier_empty.onnx .
```

Note that it uses Python 3.13.

The docker image is published to agrigorev/model-2025-hairstyle:v1.

A few notes:

The image already contains a model and it's not the same model as the one we used for questions 1-4.

## Question 5

Download the base image agrigorev/model-2025-hairstyle:v1. You can do it with docker pull.

So what's the size of this base image?

- 88 Mb
- 208 Mb
- 608 Mb
- 1208 Mb

You can get this information when running docker images - it'll be in the "SIZE" column.


In [37]:
! docker image ls | grep agrigorev/model-2025-hairstyle:v1

agrigorev/model-2025-hairstyle:v1                     4528ad1525d5        608MB             0B        


A/ `608 Mb`

## Question 6

Now let's extend this docker image, install all the required libraries and add the code for lambda.

You don't need to include the model in the image. It's already included. The name of the file with the model is hair_classifier_empty.onnx and it's in the current workdir in the image (see the Dockerfile above for the reference). The provided model requires the same preprocessing for images regarding target size and rescaling the value range than used in homework 8.

Now run the container locally.

Score this image: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

What's the output from the model?

- -1.0
- -0.10
- 0.10
- 1.0


In [39]:
import requests

url = 'http://localhost:8080/2015-03-31/functions/function/invocations'

request = {
    "url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
}

result = requests.post(url, json=request).json()
print(result)

{'curly': -0.10220833122730255}


A/ `-0.10`